In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 72
from IPython.display import display
import numpy as np
from scipy.sparse import lil_matrix
import os
import pandas as pd

ARTICLES_FOLDER= '../../datasets/newsarticles/'

In [2]:
df = pd.DataFrame()
for f in sorted([_ for _ in os.listdir(ARTICLES_FOLDER) if _.startswith('Articles') and _.endswith('.csv')]):
    dftemp = pd.read_csv(os.path.join(ARTICLES_FOLDER,f))
    df = pd.concat([df, dftemp])

# Sanity
print(f'#rows={len(df)} #columns={len(df.columns)}')
df.head()

#rows=9335 #columns=16


,abstract,articleID,articleWordCount,byline,documentType,headline,keywords,multimedia,newDesk,printPage,pubDate,sectionName,snippet,source,typeOfMaterial,webURL
0,NaN,58def1347c459f24986d7c80,716,By STEPHEN HILTNER and SUSAN LEHMAN,article,Finding an Expansive View of a Forgotten Peop...,"['Photography', 'New York Times', 'Niger', 'Fe...",3,Insider,2,2017-04-01 00:15:41,Unknown,One of the largest photo displays in Times his...,The New York Times,News,https://www.nytimes.com/2017/03/31/insider/nig...
1,NaN,58def3237c459f24986d7c84,823,By GAIL COLLINS,article,"And Now, the Dreaded Trump Curse","['United States Politics and Government', 'Tru...",3,OpEd,23,2017-04-01 00:23:58,Unknown,Meet the gang from under the bus.,The New York Times,Op-Ed,https://www.nytimes.com/2017/03/31/opinion/and...
2,NaN,58def9f57c459f24986d7c90,575,By THE EDITORIAL BOARD,article,Venezuela’s Descent Into Dictatorship,"['Venezuela', 'Politics and Government', 'Madu...",3,Editorial,22,2017-04-01 00:53:06,Unknown,A court ruling annulling the legislature’s aut...,The New York Times,Editorial,https://www.nytimes.com/2017/03/31/opinion/ven...
3,NaN,58defd317c459f24986d7c95,1374,By MICHAEL POWELL,article,Stain Permeates Basketball Blue Blood,"['Basketball (College)', 'University of North ...",3,Sports,1,2017-04-01 01:06:52,College Basketball,"For two decades, until 2013, North Carolina en...",The New York Times,News,https://www.nytimes.com/2017/03/31/sports/ncaa...
4,NaN,58df09b77c459f24986d7ca7,708,By DEB AMLEN,article,Taking Things for Granted,['Crossword Puzzles'],3,Games,0,2017-04-01 02:00:14,Unknown,In which Howard Barkin and Will Shortz teach u...,The New York Times,News,https://www.nytimes.com/2017/03/31/crosswords/...


In [3]:
Headlines = [_ for _ in df.headline if _ != 'Unknown']

# sanity
print(len(Headlines))
print(Headlines[0:3])

8603
['Finding an Expansive View  of a Forgotten People in Niger', 'And Now,  the Dreaded Trump Curse', 'Venezuela’s Descent Into Dictatorship']


In [4]:
from nltk import regexp_tokenize

# from the sklearn API the default token_pattern='(?u)\\b\\w\\w+\\b'
TOKEN_PATTERN= r'(?u)\b[a-zA-Z]\w+\b'

Headlines_tok = [regexp_tokenize(_.lower(), TOKEN_PATTERN) for _ in Headlines]

In [5]:
class Markov:  # Uses a sparse matrix to handle probabilities
    def __init__(self, order=1):
        self.voc, self.vocindex, self.vocindexrev = None, None, None
        self.conds, self.condsindex, self.condsindexrev = None, None, None
        self.Pjoint = None  # Joint probabilities
        self.Pvoc = None  # marginal P for vocabulary
        self.Pconds = None  # marginal P for conditionings
        self.Pconditional = None  # Conditional probabilities, Markov model
        self.order = order

    def get_vocabulary(self, _toks):  # input as list of tokens, 1-gram
        voc = set()
        for _ in _toks:
            voc.update(_)
        self.voc = sorted(voc)
        self.vocindex = {v:i for i, v in enumerate(self.voc)}
        self.vocindexrev = {i:v for i, v in enumerate(self.voc)}
    
    def get_conds(self, _toks):  # conditioning events, n-1-gram
        conds = set()
        for tok in _toks:
            for i in range(len(tok)-self.order):  # boundary condition
                cond = " ".join(tok[i:i+self.order])
                conds.update([cond])
        self.conds = sorted(conds)
        self.condsindex = {cond:i for i, cond in enumerate(self.conds)}
        self.condsindexrev = {i:cond for i, cond in enumerate(self.conds)}
    
    def get_counts(self, _toks):  # build the P
        M, N = len(self.vocindex), len(self.condsindex)
        pc = lil_matrix((N,M), dtype=np.float32)
        # pc = np.zeros((N,M), dtype=np.float32)
        for tok in _toks:
            for i in range(len(tok)-self.order-1):  # boundary condition
                cond = " ".join(tok[i:i+self.order])
                voc = tok[i+self.order]
                i, j = self.condsindex[cond], self.vocindex[voc]
                pc[i,j] += 1
        return pc

    def fit(self, _toks):
        self.get_vocabulary(_toks)
        self.get_conds(_toks)
        pc = self.get_counts(_toks)
        # joint P, make it into probabilities
        self.Pjoint = pc / np.sum(pc)
        # marginal P for vocabulary and conds
        self.Pvoc = np.array(np.sum(self.Pjoint,axis=0)).squeeze()
        self.Pconds = np.array(np.sum(self.Pjoint,axis=1)).squeeze()
        # conditional P
        sm = np.array(pc.sum(axis=1)).squeeze()
        sm[sm==0] = 1  # handle divide by zero
        self.Pconditional = lil_matrix(pc / sm[:,None])
        # sanity
        print(f'Size of vocabulary={len(self.voc)}, N={len(self.conds)}, Pjoint.shape={self.Pjoint.shape}')
        return self

In [6]:
markov1 = Markov(order=1).fit(Headlines_tok)

Size of vocabulary=10455, N=8942, Pjoint.shape=(8942, 10455)


In [18]:
markov3 = Markov(order=3).fit(Headlines_tok)

def getwords(_model, _w):
    i1 = _model.condsindex[_w]
    arr = _model.Pconditional[i1,:].toarray().squeeze()  # argsort will not work on lil_matrix
    ix = np.argsort(arr)[::-1]
    return [(_model.vocindexrev[ix[_]], _model.Pconditional[_model.condsindex[_w],ix[_]]) for _ in range(10)]
    
SEQ = "to be or"
results = getwords(markov3, SEQ)

for w, p in results:
    print(f"{SEQ} {w} (P={p:.5f})")

Size of vocabulary=10455, N=29775, Pjoint.shape=(29775, 10455)
to be or not (P=1.00000)
to be or zucktown (P=0.00000)
to be or finishes (P=0.00000)
to be or fireworks (P=0.00000)
to be or firewall (P=0.00000)
to be or fires (P=0.00000)
to be or firepower (P=0.00000)
to be or fired (P=0.00000)
to be or fire (P=0.00000)
to be or finn (P=0.00000)
